# vcfclick — a hands-on demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nuin/vcfclick/blob/main/examples/vcfclick-demo.ipynb)

[vcfclick](https://github.com/nuin/vcfclick) turns VCF cohorts into local,
queryable SQL databases — with trio/family analysis, gnomAD rarity filtering,
sample QC, and a GATK3-`CombineVariants` reimplementation. No server, no daemon.

This notebook **downloads its own data** and runs end to end (Colab or local
Jupyter). It doubles as a release smoke-test. Each cell shows the real output
of a v0.7.0 run.

> Trio analysis here is validated against the **GIAB Ashkenazi trio** benchmark
> — see [docs/VALIDATION.md](https://github.com/nuin/vcfclick/blob/main/docs/VALIDATION.md).

## Install + fetch the demo data

In [ ]:
# In Colab / a fresh environment, install the package (+ web UI extra):
!pip install -q "vcfclick[web]"
!vcfclick --version

vcfclick, version 0.7.0


In [ ]:
import os, urllib.request, pathlib

# Throwaway sandbox so we never touch a real ~/.vcfclick
os.environ["VCFCLICK_HOME"] = "/tmp/vcfclick-demo"
os.environ["VCFCLICK_ANNOTATIONS_DB"] = "/tmp/vcfclick-demo/ann.duckdb"
pathlib.Path("/tmp/vcfclick-demo").mkdir(parents=True, exist_ok=True)

# Download the example VCFs straight from the public repo
RAW = "https://raw.githubusercontent.com/nuin/vcfclick/main/tests/fixtures/"
FILES = [
    "tiny.vcf.gz", "qc_sex.vcf.gz", "qc_sex.ped",
    "gnomad_cftr.vcf.gz", "callset_a.vcf", "callset_b.vcf",
    "giab/cftr_trio.vcf.gz", "giab/cftr_trio.ped",
    "giab/denovo_trio.vcf.gz", "giab/denovo_trio.ped",
]
data = pathlib.Path("data"); data.mkdir(exist_ok=True)
for f in FILES:
    dest = data / pathlib.Path(f).name
    urllib.request.urlretrieve(RAW + f, dest)
print("downloaded", len(FILES), "files into ./data")

downloaded 10 files into ./data


## 1. A cohort in, SQL out

Ingest a VCF and query it. Variants, genotypes, and samples land in an embedded
ClickHouse (chDB) database; queries run in-process.

In [ ]:
!vcfclick db create demo
!vcfclick db ingest demo data/tiny.vcf.gz --cohort study --ingest-id v1 --serial
!vcfclick db query demo "SELECT chrom, pos, ref, alt FROM variants ORDER BY pos"

created  demo  ->  /tmp/vcfclick-demo/dbs/demo
[ingest] done. 5 variants in 0.3s (16/s)
   ┌─chrom─┬──pos─┬─ref─┬─alt─┐
1. │ chr1  │  100 │ A   │ G   │
2. │ chr1  │  250 │ C   │ T   │
3. │ chr1  │  500 │ G   │ A   │
   └───────┴──────┴─────┴─────┘


## 2. Trio de novo — and why it's *defensible*

De novo means the child carries a variant **neither parent has** — which needs
the parents to be *provably* hom-reference, not merely absent. vcfclick's
genotype table is sparse (a missing call is `0/0` **or** `./.`), so de novo
requires a `--keep-reference` ingest that stores confident parent `0/0`.

The fixture is the **real GIAB trio** (HG002 son, HG003 father, HG004 mother)
at 7 chr20 sites: 4 are confident de novos (both parents in GIAB's
high-confidence BED), 3 have a no-call parent.

In [ ]:
!vcfclick db create dn
!vcfclick db ingest dn data/denovo_trio.vcf.gz --cohort fam --ingest-id g1 --serial --keep-reference
!vcfclick db ped  dn data/denovo_trio.ped
!vcfclick db trio dn --proband HG002 --category denovo --max-af 1.0

loaded pedigree: 3 individuals under ingest_id=g1
trio: proband=HG002 father=HG003 mother=HG004

denovo candidates: 4
  chr20:2202238 C>T  proband_gt=1 father_gt=0 mother_gt=0  AF=NA
  chr20:5739093 A>G  proband_gt=1 father_gt=0 mother_gt=0  AF=NA
  chr20:5864738 G>A  proband_gt=1 father_gt=0 mother_gt=0  AF=NA
  chr20:5893991 G>C  proband_gt=1 father_gt=0 mother_gt=0  AF=NA


Exactly the **4** confident sites — the 3 no-call-parent sites are excluded.
`bcftools +mendelian2` and `slivar` independently flag the same 4. Re-ingest the
*same data* **without** `--keep-reference` and de novo correctly returns **0**:
vcfclick refuses to guess where a parent is a no-call.

In [ ]:
!vcfclick db create dn_naive
!vcfclick db ingest dn_naive data/denovo_trio.vcf.gz --cohort fam --ingest-id g1 --serial
!vcfclick db ped  dn_naive data/denovo_trio.ped
!vcfclick db trio dn_naive --proband HG002 --category denovo --max-af 1.0 2>/dev/null | grep candidates

denovo candidates: 0


## 3. Compound-het + gnomAD population-frequency filtering

A real CFTR slice of the GIAB trio reproduces every inheritance model. CFTR is a
genuine **compound-het** candidate gene — a paternal-origin het plus
maternal-origin hets in *trans*. Loading real gnomAD frequencies and adding
`--gnomad-max-af` then drops the variants that are actually common in the
population.

In [ ]:
!vcfclick db create fam
!vcfclick db ingest fam data/cftr_trio.vcf.gz --cohort trio --ingest-id g1 --serial --keep-reference
!vcfclick db ped  fam data/cftr_trio.ped
!vcfclick annotations load-gnomad data/gnomad_cftr.vcf.gz
# CFTR gene coords (normally `vcfclick annotations load` pulls all of GENCODE):
!python -c "from annotations.db import get_connection as g; c=g(); c.execute(\"INSERT INTO refseq_genes VALUES ('CFTR','chr7',117480025,117668665,'+','1080','x')\"); c.close()"

print('\n--- all models ---')
!vcfclick db trio fam --proband HG002
print('\n--- + gnomAD popmax < 0.01 ---')
!vcfclick db trio fam --proband HG002 --gnomad-max-af 0.01

loaded   5 gnomAD allele frequencies into the annotation store

--- all models ---
  denovo          0
  recessive       2
  dominant        3
  comphet         1  genes

--- + gnomAD popmax < 0.01 ---
  denovo          0
  recessive       0
  dominant        1
  comphet         0  genes


The CFTR recessive sites are common in gnomAD (popmax ≈ 0.8) so they drop out;
one dominant het is genuinely rare (popmax 0.009) and survives. **Defensible
rarity from true population data**, not the cohort's own frequencies.

## 4. Per-sample QC + a chrX sex check

`db qc` reports het/hom, Ti/Tv, and infers sex from chrX heterozygosity —
flagging it against the pedigree (the sample-swap signal). Here we read the
JSON straight into a DataFrame.

In [ ]:
import json, subprocess, pandas as pd
!vcfclick db create qc
!vcfclick db ingest qc data/qc_sex.vcf.gz --cohort c --ingest-id i1 --serial
!vcfclick db ped  qc data/qc_sex.ped
out = subprocess.run(["vcfclick","db","qc","qc","--format","json"],
                     capture_output=True, text=True).stdout
pd.DataFrame(json.loads(out))[
    ["sample_id","het","hom_alt","het_hom_ratio","ti_tv","chrx_het_frac","inferred_sex","sex_mismatch"]
]

  sample_id  het  hom_alt het_hom_ratio ti_tv  chrx_het_frac inferred_sex  sex_mismatch
0        F1   12       13          0.92  1.08          0.480       female         False
1        M1    1       24          0.04  1.08          0.040         male         False
2      SWAP   12       13          0.92  1.08          0.480       female          True

`SWAP` infers **female** but the pedigree declares it **male** → `sex_mismatch =
True`, the classic sample-swap flag.

## 5. Combine call sets — the GATK3 `CombineVariants` GATK4 removed

`combine` unions call sets that may *share* samples (two callers over one
cohort), resolving overlaps by input priority and annotating each record with
`set=` provenance.

In [ ]:
!vcfclick combine data/callset_a.vcf data/callset_b.vcf -o /tmp/combined.vcf --name first --name second
!grep -v '^##' /tmp/combined.vcf

combined → /tmp/combined.vcf
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	S1	S2	S3
chr1	100	.	A	T	.	.	set=first	GT	0/1	0/0	./.
chr1	200	.	C	G	.	.	set=Intersection	GT	1/1	0/1	0/1
chr1	300	.	G	A	.	.	set=Intersection	GT	0/1	0/1	0/0
chr1	400	.	T	C	.	.	set=second	GT	./.	0/0	1/1


`set=` shows each record's source (`Intersection` when in all), the shared
sample `S2` is resolved by priority, and an absent sample is `./.` — never a
silent `0/0`. This is verified byte-equivalent to real GATK 3.8 output.

## 6. A picture of the defensibility win

Across a 5 Mb chr20 region of the GIAB trio, the naive "neither parent carries"
rule calls **50** de novos; GIAB's high-confidence BED confirms only **4**.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["naive\nde novo", "confirmed\n(GIAB BED)"], [50, 4],
       color=["#f59e0b", "#16a34a"])
ax.set_ylabel("de-novo calls in 5 Mb")
ax.set_title("vcfclick --keep-reference avoids ~46 false positives")
for i, v in enumerate([50, 4]):
    ax.text(i, v + 1, str(v), ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

## Next steps

- **Web UI:** `vcfclick web demo` — a local SQL explorer + natural-language→SQL
  + trio/combine panels.
- **Docs:** [README](https://github.com/nuin/vcfclick) ·
  [Trio](https://github.com/nuin/vcfclick/blob/main/docs/TRIO.md) ·
  [Validation](https://github.com/nuin/vcfclick/blob/main/docs/VALIDATION.md)
- **Install:** `uv tool install vcfclick` (or `pip install "vcfclick[web]"`).